# Hilbert Spaces: Exercises

This notebook contains 8 progressive exercises covering inner products, Hilbert norms, projection, least squares, orthonormal bases, Riesz representation, operators, and kernel previews. Each exercise has a working area followed by a full solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(11)

## Exercise 1: Check an Inner Product

Let $W=\begin{bmatrix}2 & 1\\ 1 & 2\end{bmatrix}$. Verify numerically that $\langle \mathbf{x},\mathbf{y}\rangle_W=\mathbf{x}^\top W\mathbf{y}$ is symmetric and positive definite on several random vectors.

In [ ]:
# Your Solution
W = np.array([[2.0, 1.0], [1.0, 2.0]])
print("Try checking symmetry and positive definiteness for random vectors.")

In [ ]:
# Solution
W = np.array([[2.0, 1.0], [1.0, 2.0]])
eigvals = np.linalg.eigvalsh(W)
assert np.all(eigvals > 0)

for _ in range(10):
    x = rng.normal(size=2)
    y = rng.normal(size=2)
    left = x @ W @ y
    right = y @ W @ x
    assert np.isclose(left, right)
    assert x @ W @ x > 0

print("W is positive definite, so the weighted form is an inner product.")

---

## Exercise 2: Parallelogram Law

Show with a concrete counterexample that $\lVert \cdot \rVert_1$ on $\mathbb{R}^2$ is not induced by an inner product.

In [ ]:
# Your Solution
u = np.array([1.0, 0.0])
v = np.array([0.0, 1.0])
print("Compute both sides of the parallelogram law using the L1 norm.")

In [ ]:
# Solution
def l1(x):
    return np.linalg.norm(x, ord=1)

u = np.array([1.0, 0.0])
v = np.array([0.0, 1.0])
lhs = l1(u + v) ** 2 + l1(u - v) ** 2
rhs = 2 * (l1(u) ** 2 + l1(v) ** 2)
print("lhs =", lhs, "rhs =", rhs)
assert lhs != rhs
print("The L1 norm fails the parallelogram law, so it is not a Hilbert norm.")

---

## Exercise 3: Projection Onto a Subspace

Project $\mathbf{x}=(3,2,1)$ onto the span of $\mathbf{a}=(1,1,0)$ and $\mathbf{b}=(0,1,1)$. Verify the residual is orthogonal to both spanning vectors.

In [ ]:
# Your Solution
x = np.array([3.0, 2.0, 1.0])
A = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
print("Use P = A(A^T A)^{-1}A^T or solve least squares.")

In [ ]:
# Solution
x = np.array([3.0, 2.0, 1.0])
A = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
coef = np.linalg.solve(A.T @ A, A.T @ x)
projection = A @ coef
residual = x - projection

print("coefficients:", coef)
print("projection:", projection)
print("A^T residual:", A.T @ residual)
assert np.allclose(A.T @ residual, 0.0)

---

## Exercise 4: Least Squares as Orthogonal Projection

Fit a line $y=\beta_0+\beta_1 t$ to four points and verify $X^\top\mathbf{r}=\mathbf{0}$.

In [ ]:
# Your Solution
t = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 2.9, 4.2])
print("Build X with columns [1, t], solve least squares, then inspect X.T @ residual.")

In [ ]:
# Solution
t = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 2.9, 4.2])
X = np.column_stack([np.ones_like(t), t])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
residual = y - X @ beta

print("beta:", beta)
print("X^T residual:", X.T @ residual)
assert np.allclose(X.T @ residual, 0.0)

---

## Exercise 5: Gram-Schmidt and Parseval

Orthonormalize three independent vectors and verify Parseval identity for a test vector.

In [ ]:
# Your Solution
V = np.array([[1.0, 1.0, 0.0], [0.0, 1.0, 1.0], [1.0, 0.0, 1.0]])
z = np.array([2.0, -1.0, 0.5])
print("Construct Q, compute Q.T @ Q, and compare norm(z)^2 with sum of squared coordinates.")

In [ ]:
# Solution
def gram_schmidt(V):
    cols = []
    for v in V.T:
        u = v.astype(float).copy()
        for q in cols:
            u -= (q @ u) * q
        cols.append(u / np.linalg.norm(u))
    return np.column_stack(cols)

V = np.array([[1.0, 1.0, 0.0], [0.0, 1.0, 1.0], [1.0, 0.0, 1.0]])
z = np.array([2.0, -1.0, 0.5])
Q = gram_schmidt(V)
coords = Q.T @ z

print("Q^T Q:")
print(Q.T @ Q)
print("norm squared:", z @ z)
print("coordinate energy:", coords @ coords)
assert np.allclose(Q.T @ Q, np.eye(3))
assert np.allclose(z @ z, coords @ coords)

---

## Exercise 6: Riesz Representative in Weighted Geometry

For $L(\mathbf{x})=\mathbf{a}^\top\mathbf{x}$ and $\langle\mathbf{x},\mathbf{y}\rangle_G=\mathbf{x}^\top G\mathbf{y}$, find the representative $\mathbf{h}$ satisfying $L(\mathbf{x})=\langle\mathbf{x},\mathbf{h}\rangle_G$.

In [ ]:
# Your Solution
G = np.array([[3.0, 1.0], [1.0, 2.0]])
a = np.array([4.0, -1.0])
print("Solve G h = a.")

In [ ]:
# Solution
G = np.array([[3.0, 1.0], [1.0, 2.0]])
a = np.array([4.0, -1.0])
h = np.linalg.solve(G, a)

for _ in range(5):
    x = rng.normal(size=2)
    assert np.isclose(a @ x, x @ G @ h)

print("Riesz representative:", h)

---

## Exercise 7: Positive Self-Adjoint Operator

Create a covariance matrix from data. Verify it is symmetric positive semidefinite and interpret the largest eigenvector as the first PCA direction.

In [ ]:
# Your Solution
data = rng.normal(size=(80, 3))
print("Center the data, form C = X.T @ X / n, and diagonalize C.")

In [ ]:
# Solution
data = rng.normal(size=(80, 3)) @ np.array([[2.0, 0.0, 0.0], [0.5, 1.0, 0.0], [0.0, 0.2, 0.3]])
centered = data - data.mean(axis=0)
C = centered.T @ centered / len(centered)
vals, vecs = np.linalg.eigh(C)
order = np.argsort(vals)[::-1]
vals = vals[order]
vecs = vecs[:, order]

print("eigenvalues:", vals)
print("first PCA direction:", vecs[:, 0])
assert np.allclose(C, C.T)
assert vals[-1] > -1e-12

---

## Exercise 8: Kernel Gram Matrix Preview

Build an RBF kernel Gram matrix for one-dimensional points and verify it is positive semidefinite up to numerical precision.

In [ ]:
# Your Solution
xs = np.linspace(-1.5, 1.5, 10)[:, None]
print("Compute K_ij = exp(-||x_i - x_j||^2 / (2 sigma^2)) and inspect eigenvalues.")

In [ ]:
# Solution
xs = np.linspace(-1.5, 1.5, 10)[:, None]
sigma = 0.7
sqdist = (xs - xs.T) ** 2
K = np.exp(-sqdist / (2 * sigma**2))
eigvals = np.linalg.eigvalsh(K)

print("kernel eigenvalues:", eigvals)
assert eigvals[0] > -1e-10
print("The finite Gram matrix is PSD, consistent with an implicit Hilbert feature space.")

## Closing Reflection

The exercises move from axioms to geometry, then from geometry to ML objects. That is the right mental order: first know what an inner product gives you, then recognize where algorithms are using it.